In [1]:
import pandas as pd
import requests
import io

In [2]:
from src.ecb_api import get_ecb_series

# Short name for each series -> its key on the ECB Data Portal
SERIES = {
    "DFR": "FM.D.U2.EUR.4F.KR.DFR.LEV",      # Deposit facility rate (daily)
    "ESTR": "EST.B.EU000A2X2A25.WT",         # Euro short-term rate (business days)
    "HICP": "HICP.M.U2.N.000000.4D0.ANR",    # Headline inflation, annual rate (monthly)
    "CORE": "HICP.M.U2.N.XEF000.4D0.ANR",    # Core inflation, annual rate (monthly)
}

In [3]:
# Download every series and store it in a dictionary
data = {}

for name, key in SERIES.items():
    series = get_ecb_series(key)
    data[name] = series
    print(f"{name:5} {len(series):5} obs.  from {series.index.min().date()}  to {series.index.max().date()}")# Download every series and store it in a dictionary

DFR    4292 obs.  from 2015-01-01  to 2026-10-01
ESTR   1793 obs.  from 2019-10-01  to 2026-09-30
HICP    140 obs.  from 2015-01-01  to 2026-08-01
CORE    140 obs.  from 2015-01-01  to 2026-08-01


Downloading the data through the API makes the analysis reproducible and easy to update. When a new monthly figure is released, re-running the notebook fetches the latest data automatically, with no manual copying that could introduce errors. It also documents exactly which series and keys were used, so anyone can replicate the results.

In [4]:
for name, series in data.items():
    print(name)
    print("Observations", len(series))
    print("First date: ", series.index.min().date())
    print("Last date: ", series.index.max().date())
    print("Missing values: ", series.isna().sum())

DFR
Observations 4292
First date:  2015-01-01
Last date:  2026-10-01
Missing values:  0
ESTR
Observations 1793
First date:  2019-10-01
Last date:  2026-09-30
Missing values:  0
HICP
Observations 140
First date:  2015-01-01
Last date:  2026-08-01
Missing values:  0
CORE
Observations 140
First date:  2015-01-01
Last date:  2026-08-01
Missing values:  0


In [5]:
data["DFR"].describe()


count    4292.000000
mean        0.693628
std         1.602772
min        -0.500000
25%        -0.400000
50%        -0.400000
75%         2.000000
max         4.000000
Name: FM.D.U2.EUR.4F.KR.DFR.LEV, dtype: float64

In [6]:
dfr_monthly = data["DFR"].resample("MS").mean()
estr_monthly = data["ESTR"].resample("MS").mean()

In [ ]:
monthly = pd.concat([dfr_monthly, estr_monthly, data["HICP"], data["CORE"]], axis=1)
monthly.columns = ["DFR", "ESTR", "HICP", "CORE"]

monthly.isna().sum()
monthly = monthly.dropna(subset = ["HICP"])




DFR      0
ESTR    57
HICP     0
CORE     0
dtype: int64

In [21]:
changes = data["DFR"].diff()
changes = changes[changes != 0].dropna()
print(changes)

print("Total changes:", len(changes))
print("Hikes:", (changes > 0).sum())
print("Cuts:", (changes < 0).sum())


TIME_PERIOD
2015-12-09   -0.10
2016-03-16   -0.10
2019-09-18   -0.10
2022-07-27    0.50
2022-09-14    0.75
2022-11-02    0.75
2022-12-21    0.50
2023-02-08    0.50
2023-03-22    0.50
2023-05-10    0.25
2023-06-21    0.25
2023-08-02    0.25
2023-09-20    0.25
2024-06-12   -0.25
2024-09-18   -0.25
2024-10-23   -0.25
2024-12-18   -0.25
2025-02-05   -0.25
2025-03-12   -0.25
2025-04-23   -0.25
2025-06-11   -0.25
2026-06-17    0.25
2026-09-16    0.25
Name: FM.D.U2.EUR.4F.KR.DFR.LEV, dtype: float64
Total changes: 23
Hikes: 12
Cuts: 11


We use the monthly average of the DFR rather than the end-of-month value due to the fact that the HICP is a monthly measure: it reflects average prices over the whole month. Using the monthly average of the DFR makes both series comparable, because it captures the rate that was actually in force during the month. When the ECB changes the rate in the middle of a month, the average reflects that only part of the month was at the new level, whereas the end-of-month value would assign the new rate to the whole month. In practice the difference is small, since the DFR only changes a few times a year, and it only affects the months in which a decision took effect.

Since January 2015, the DFR has changed 23 times: 12 hikes and 11 cuts. The changes are clustered in three phases: cuts into negative territory until 2019, a rapid tightening cycle in 2022–2023 in response to the inflation surge, and a gradual easing cycle from 2024 onwards.